# Module 07: TensorFlow & Keras for Deep Learning
## Notebook 05: Convolutional Neural Networks & Transfer Learning

Convolutional Neural Networks (CNNs) revolutionized computer vision by introducing spatial weight sharing and translation invariance. Instead of treating pixels as independent features, CNNs learn hierarchical spatial filters that detect edges, textures, patterns, and high-level semantic objects.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Master core 2D Convolution mechanics: receptive fields, filters, strides, and padding (`valid` vs `same`).
2. Aggregate spatial representations using `MaxPooling2D` and parameter-efficient `GlobalAveragePooling2D`.
3. Build a modern deep CNN from scratch with in-graph data augmentations (`RandomFlip`, `RandomRotation`).
4. **Advanced:** Implement **Transfer Learning and Fine-Tuning** using a pretrained deep convolutional backbone.
5. **Advanced:** Extract and visualize **Intermediate Feature Activation Maps** to inspect what neural networks learn.

In [ ]:
import os
import tensorflow as tf
import keras
from keras import layers, models, optimizers, losses, metrics
import numpy as np
import matplotlib.pyplot as plt

img_dir = "images" if os.path.exists("images") else "../images"
print(f"TensorFlow Version: {tf.__version__}")

### 1. Convolutional Layer Mechanics
- **`layers.Conv2D(filters, kernel_size, strides, padding)`:**
  - `filters`: Number of output feature maps (e.g. 32, 64).
  - `kernel_size`: Size of sliding window (typically $3 \times 3$ or $5 \times 5$).
  - `padding="same"`: Zero-pads the input border so output spatial dimensions match input dimensions.
  - `padding="valid"`: No padding; output shrinks by $(\text{kernel\_size} - 1)$.
- **`layers.MaxPooling2D(pool_size=(2, 2))`:** Downsamples feature maps by taking the maximum value in non-overlapping $2 \times 2$ windows, halving height and width while preserving dominant activations.

In [ ]:
# Build a modern Modular CNN from scratch
def create_modular_cnn(input_shape=(64, 64, 3), num_classes=3):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # In-graph data augmentations
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),

        # Conv Block 1
        layers.Conv2D(32, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),

        # Conv Block 2
        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),

        # Conv Block 3
        layers.Conv2D(128, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        # Global feature aggregation (eliminates massive parameter count of Flatten)
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ], name="Modular_Vision_CNN")
    return model

cnn = create_modular_cnn()
cnn.summary()

### 2. Training CNN on Synthetic Geometric Vision Benchmark
We synthesize a multi-class vision dataset (Circles, Squares, Triangles) with random variations and train our CNN model:

In [ ]:
# Synthesize multi-class vision dataset (N=600 images of size 64x64x3)
import cv2

np.random.seed(42)
N_samples = 600
X_synth = np.full((N_samples, 64, 64, 3), 230, dtype=np.uint8)
y_synth = np.zeros(N_samples, dtype=np.int32)

for i in range(N_samples):
    cls = i % 3
    y_synth[i] = cls
    color = (np.random.randint(40, 200), np.random.randint(40, 200), np.random.randint(40, 200))
    center = (np.random.randint(24, 40), np.random.randint(24, 40))
    radius = np.random.randint(12, 18)

    if cls == 0: # Circle
        cv2.circle(X_synth[i], center, radius, color, -1)
    elif cls == 1: # Square
        cv2.rectangle(X_synth[i], (center[0]-radius, center[1]-radius), (center[0]+radius, center[1]+radius), color, -1)
    elif cls == 2: # Triangle
        pts = np.array([[center[0], center[1]-radius], [center[0]-radius, center[1]+radius], [center[0]+radius, center[1]+radius]], np.int32)
        cv2.fillPoly(X_synth[i], [pts], color)

# Normalize pixel values to [0.0, 1.0]
X_norm = (X_synth / 255.0).astype(np.float32)
y_onehot = keras.utils.to_categorical(y_synth, num_classes=3)

# Train CNN
cnn.compile(
    optimizer=optimizers.Adam(learning_rate=0.002),
    loss=losses.CategoricalCrossentropy(),
    metrics=["accuracy"]
)

history_cnn = cnn.fit(
    X_norm, y_onehot,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

### 3. Complex Application 1: Transfer Learning & Fine-Tuning Workflow
Training deep models from scratch requires millions of labeled images. **Transfer Learning** reuses weights learned on massive foundation datasets (ImageNet):
- **Stage 1 (Feature Extraction):** Freeze the pretrained backbone (`base_model.trainable = False`). Train only the newly appended task-specific classification head.
- **Stage 2 (Fine-Tuning):** Unfreeze top convolutional blocks in the backbone and continue training with a very small learning rate ($\eta = 10^{-5}$) to gently adapt high-level representations without destroying pre-learned features.

In [ ]:
# Construct Transfer Learning Architecture using a compact MobileNetV2 backbone
base_model = keras.applications.MobileNetV2(
    input_shape=(64, 64, 3),
    include_top=False,
    weights="imagenet"
)

# Stage 1: Freeze base model
base_model.trainable = False

inputs = layers.Input(shape=(64, 64, 3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(3, activation="softmax")(x)

tl_model = models.Model(inputs=inputs, outputs=outputs, name="Transfer_Learning_MobileNet")
tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"Total Base Model Layers: {len(base_model.layers)}")
print(f"Trainable Weights (Head Only): {len(tl_model.trainable_weights)}")

# Train feature extraction head for 2 epochs
tl_model.fit(X_norm, y_onehot, epochs=2, batch_size=32, verbose=1)

# Stage 2: Fine-Tuning (Unfreeze the top 20 layers of the backbone)
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5), # Reduced learning rate for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print(f"Trainable Weights after unfreezing top blocks: {len(tl_model.trainable_weights)}")

### 4. Complex Application 2: Inspecting Intermediate Feature Activation Maps
To understand what deep convolutional networks see:
- We extract intermediate feature maps using the Functional API.
- Early layers detect low-level edges and color gradients.
- Deeper layers assemble features into high-level geometric parts and textures.

In [ ]:
# Extract activations from the first convolutional layer of our custom CNN
conv1_layer = [layer for layer in cnn.layers if "conv2d" in layer.name][0]
activation_model = models.Model(inputs=cnn.inputs, outputs=conv1_layer.output)

# Select a single test sample (a circle)
sample_img = X_norm[0:1]
activations = activation_model.predict(sample_img, verbose=0) # Shape: (1, 64, 64, 32)

print(f"Feature activation maps shape: {activations.shape}")

# Plot 6 feature channels
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for i in range(6):
    axes[i].imshow(activations[0, :, :, i], cmap="viridis")
    axes[i].set_title(f"Channel {i+1}")
    axes[i].axis("off")

plt.suptitle("First Convolutional Layer Feature Maps (Low-Level Edge & Corner Filters)", y=1.05)
plt.tight_layout()
plt.show()